# Battery deployment 2026 vs 2025

In [1]:
import pandas as pd
import pathlib
import mlflow
import sklearn.neighbors
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [2]:
def read_data(rel_path):
    path = pathlib.Path("../data").resolve() / (rel_path + ".parquet")
    df = pd.read_parquet(path)
    return df

In [3]:
data_sessions_2026 = read_data("gold/session_metadata_Y2026")
data_sessions_2025 = read_data("gold/session_metadata_Y2025")


In [4]:
data_sessions_2026[['round_number', 'event_location']].drop_duplicates()

,round_number,event_location
0,1,Melbourne
5,2,Shanghai
10,3,Suzuka
15,4,Miami Gardens
20,0,Bahrain


In [5]:
data_sessions_2025[['round_number', 'event_location']].drop_duplicates()

,round_number,event_location
0,1,Melbourne
5,2,Shanghai
10,3,Suzuka
15,4,Sakhir
20,5,Jeddah
25,6,Miami Gardens
30,7,Imola
35,8,Monaco
40,9,Barcelona
45,10,Montréal


In [6]:
data_laps_2026 = read_data("gold/session_laps_Y2026")
data_laps_2025 = read_data("gold/session_laps_Y2025")


In [7]:
data_telemetry_pos_2026 = read_data("gold/telemetry_pos_Y2026R02")
data_telemetry_pos_2025 = read_data("gold/telemetry_pos_Y2025R02")
data_telemetry_car_2026 = read_data("gold/telemetry_car_Y2026R02")
data_telemetry_car_2025 = read_data("gold/telemetry_car_Y2025R02")


In [8]:
data_telemetry_pos_2025

,year,session_id,driver_number,lap_number,timestamp,timing_from_session,timing_from_lap,track_status,coordinate_x,coordinate_y,coordinate_z,position_status
0,2025,Y2025R02S1,1,1,2025-03-21 03:30:56.904000+00:00,924.556,0.038,1,-3008.0,-2073.0,0.0,OnTrack
1,2025,Y2025R02S1,1,1,2025-03-21 03:30:57.184000+00:00,924.836,0.318,1,-3033.0,-2079.0,0.0,OnTrack
2,2025,Y2025R02S1,1,1,2025-03-21 03:30:57.344000+00:00,924.996,0.478,1,-3054.0,-2084.0,0.0,OnTrack
3,2025,Y2025R02S1,1,1,2025-03-21 03:30:57.744000+00:00,925.396,0.878,1,-3082.0,-2091.0,0.0,OnTrack
4,2025,Y2025R02S1,1,1,2025-03-21 03:30:57.964000+00:00,925.616,1.098,1,-3100.0,-2095.0,0.0,OnTrack
...,...,...,...,...,...,...,...,...,...,...,...,...
1147492,2025,Y2025R02S5,87,56,2025-03-23 08:35:34.411000+00:00,9058.576,95.993,1,953.0,-1291.0,142.0,OnTrack
1147493,2025,Y2025R02S5,87,56,2025-03-23 08:35:34.591000+00:00,9058.756,96.173,1,791.0,-1326.0,140.0,OnTrack
1147494,2025,Y2025R02S5,87,56,2025-03-23 08:35:34.792000+00:00,9058.957,96.374,1,578.0,-1373.0,140.0,OnTrack
1147495,2025,Y2025R02S5,87,56,2025-03-23 08:35:35.012000+00:00,9059.177,96.594,1,436.0,-1405.0,141.0,OnTrack


In [9]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
runs = mlflow.search_runs(experiment_names=["formula_one_circuit_map"])
runs = runs[runs['tags.session_ids'].str.contains('Y2026R02S1_Y2026R02S2_Y2026R02S3_Y2026R02S4_Y2026R02S5')]
print(runs.iloc[0].to_dict())
df_circuit_map_2026  =  pd.read_parquet(
    mlflow.artifacts.download_artifacts(artifact_uri=runs.iloc[0]["artifact_uri"] + "/result.parquet")
)

{'run_id': 'f5e32c9855da460384f134093aecfc04', 'experiment_id': '1', 'status': 'FINISHED', 'artifact_uri': '/Users/tiagobbatalhao/Documents/projects/formula_one_data_analysis/mlruns/1/f5e32c9855da460384f134093aecfc04/artifacts', 'start_time': Timestamp('2026-05-02 14:48:28.154000+0000', tz='UTC'), 'end_time': Timestamp('2026-05-02 14:49:38.387000+0000', tz='UTC'), 'metrics.rmse-distance-x': 2.270490581870776, 'metrics.rmse-distance-y': 2.2426098325380472, 'metrics.mae-time-y': 72.74836821252167, 'metrics.adjustment': -0.001087760021420252, 'metrics.rmse-time-x': 204.76546879620284, 'metrics.mae-distance-y': 1.3606271069229179, 'metrics.mae-time-x': 71.34567191390542, 'metrics.rmse-time-y': 167.0880340309693, 'metrics.mae-distance-x': 1.474548318082218, 'metrics.mae-time-z': 2.085532639145783, 'metrics.mae-distance-z': 0.0878034633990365, 'metrics.rmse-time-z': 5.3911086878425385, 'metrics.rmse-distance-z': 0.16347388962006165, 'params.max_degree': '100', 'params.predict_size': '100000'

In [10]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
runs = mlflow.search_runs(experiment_names=["formula_one_circuit_map"])
runs = runs[runs['tags.session_ids'].str.contains('Y2025R02S1_Y2025R02S2_Y2025R02S3_Y2025R02S4_Y2025R02S5')]
print(runs.iloc[0].to_dict())
df_circuit_map_2025  =  pd.read_parquet(
    mlflow.artifacts.download_artifacts(artifact_uri=runs.iloc[0]["artifact_uri"] + "/result.parquet")
)

{'run_id': '4a6734f93a614430b4616ead7ebbf1db', 'experiment_id': '1', 'status': 'FINISHED', 'artifact_uri': '/Users/tiagobbatalhao/Documents/projects/formula_one_data_analysis/mlruns/1/4a6734f93a614430b4616ead7ebbf1db/artifacts', 'start_time': Timestamp('2026-05-09 15:43:51.898000+0000', tz='UTC'), 'end_time': Timestamp('2026-05-09 15:45:08.506000+0000', tz='UTC'), 'metrics.rmse-distance-x': 0.5827610105859579, 'metrics.rmse-distance-y': 0.6506732202927643, 'metrics.mae-time-y': 45.73332257502941, 'metrics.adjustment': 0.00015098998089204164, 'metrics.rmse-time-x': 56.47925006327176, 'metrics.mae-distance-y': 0.3812707008752371, 'metrics.mae-time-x': 40.71767759671238, 'metrics.rmse-time-y': 68.30507735937819, 'metrics.mae-distance-x': 0.36937551701755156, 'metrics.mae-time-z': 0.540730459364819, 'metrics.mae-distance-z': 0.05312347095091729, 'metrics.rmse-time-z': 0.9627355815978573, 'metrics.rmse-distance-z': 0.10944097820707024, 'params.max_degree': '100', 'params.predict_size': '100

In [11]:
def run_circuit_encoding(telemetry_pos: pd.DataFrame, circuit_map: pd.DataFrame) -> pd.DataFrame:
    columns_coordinates = ["coordinate_x", "coordinate_y"]
    columns_pkey = ["session_id", "driver_number", "lap_number", "timestamp"]
    find_neighbours = (
        sklearn.neighbors.NearestNeighbors(n_neighbors=1)
        .fit(circuit_map[columns_coordinates].values)
        .kneighbors(telemetry_pos[columns_coordinates].values)
    )
    df_pos = telemetry_pos[columns_pkey].copy()
    df_pos["idx"] = find_neighbours[1][:, 0]
    df_pos = df_pos.merge(
        circuit_map.assign(idx=lambda df: range(len(df)))[["idx", "encoding", "distance_m"]]
    )
    return df_pos


In [12]:
if "distance_m" not in data_telemetry_pos_2026.columns:
    data_telemetry_pos_2026 = data_telemetry_pos_2026.merge(
        run_circuit_encoding(data_telemetry_pos_2026, df_circuit_map_2026)
    )
if "distance_m" not in data_telemetry_pos_2025.columns:
    data_telemetry_pos_2025 = data_telemetry_pos_2025.merge(
        run_circuit_encoding(data_telemetry_pos_2025, df_circuit_map_2025)
    )


In [13]:
lap_2026 = data_laps_2026[
    (data_laps_2026['session_id']=='Y2026R02S4')
].sort_values(by=['time_lap'], ascending=True).iloc[:1]
lap_2026.T

,4076
year,2026
session_id,Y2026R02S4
driver_number,12
driver_name,ANT
driver_team,Mercedes
lap_number,14
stint,5.0
timestamp_lap_start,2026-03-14 07:57:40.635000+00:00
timing_start_lap,4351.323
timing_end_lap,4443.387


In [14]:
lap_2025 = data_laps_2025[
    (data_laps_2025['session_id']=='Y2025R02S4')
].sort_values(by=['time_lap'], ascending=True).iloc[:1]
lap_2025.T

,3909
year,2025
session_id,Y2025R02S4
driver_number,81
driver_name,PIA
driver_team,McLaren
lap_number,19
stint,6.0
timestamp_lap_start,2025-03-22 07:58:39.876000+00:00
timing_start_lap,4294.369
timing_end_lap,4385.01


In [15]:
data_telemetry_pos_lap2026 = (
    data_telemetry_pos_2026.merge(
        lap_2026[['session_id', 'driver_number', 'lap_number']], how='inner'
    )
)
data_telemetry_car_lap2026 = (
    data_telemetry_car_2026.merge(
        lap_2026[['session_id', 'driver_number', 'lap_number']], how='inner'
    )
)

In [16]:
data_telemetry_pos_lap2025 = (
    data_telemetry_pos_2025.merge(
        lap_2025[['session_id', 'driver_number', 'lap_number']], how='inner'
    )
)
data_telemetry_car_lap2025 = (
    data_telemetry_car_2025.merge(
        lap_2025[['session_id', 'driver_number', 'lap_number']], how='inner'
    )
)

In [17]:
data_telemetry_pos_lap2026

,year,session_id,driver_number,lap_number,timestamp,timing_from_session,timing_from_lap,track_status,coordinate_x,coordinate_y,coordinate_z,position_status,idx,encoding,distance_m
0,2026,Y2026R02S4,12,14,2026-03-14 07:57:40.662000+00:00,4351.350,0.027,1,430.0,-1470.0,155.0,OnTrack,99882,0.99882,5386.661366
1,2026,Y2026R02S4,12,14,2026-03-14 07:57:40.842000+00:00,4351.530,0.207,1,209.0,-1519.0,152.0,OnTrack,305,0.00305,16.373028
2,2026,Y2026R02S4,12,14,2026-03-14 07:57:40.882000+00:00,4351.570,0.247,1,209.0,-1519.0,152.0,OnTrack,305,0.00305,16.373028
3,2026,Y2026R02S4,12,14,2026-03-14 07:57:41.222000+00:00,4351.910,0.587,1,-93.0,-1585.0,144.0,OnTrack,874,0.00874,47.273858
4,2026,Y2026R02S4,12,14,2026-03-14 07:57:41.661000+00:00,4352.349,1.026,1,-386.0,-1649.0,140.0,OnTrack,1432,0.01432,77.285708
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
351,2026,Y2026R02S4,12,14,2026-03-14 07:59:11.322000+00:00,4442.010,90.687,1,1368.0,-1266.0,145.0,OnTrack,98107,0.98107,5290.678058
352,2026,Y2026R02S4,12,14,2026-03-14 07:59:11.402000+00:00,4442.090,90.767,1,1368.0,-1266.0,144.0,OnTrack,98107,0.98107,5290.678058
353,2026,Y2026R02S4,12,14,2026-03-14 07:59:11.762000+00:00,4442.450,91.127,1,1036.0,-1339.0,141.0,OnTrack,98732,0.98732,5324.677777
354,2026,Y2026R02S4,12,14,2026-03-14 07:59:12.102000+00:00,4442.790,91.467,1,772.0,-1396.0,140.0,OnTrack,99237,0.99237,5351.668439


In [18]:
data_telemetry_pos_lap2025

,year,session_id,driver_number,lap_number,timestamp,timing_from_session,timing_from_lap,track_status,coordinate_x,coordinate_y,coordinate_z,position_status,idx,encoding,distance_m
0,2025,Y2025R02S4,81,19,2025-03-22 07:58:40.180000+00:00,4294.673,0.304,1,0.0,-1511.0,140.0,OnTrack,312,0.00312,16.753242
1,2025,Y2025R02S4,81,19,2025-03-22 07:58:40.400000+00:00,4294.893,0.524,1,-205.0,-1563.0,139.0,OnTrack,707,0.00707,37.938046
2,2025,Y2025R02S4,81,19,2025-03-22 07:58:40.800000+00:00,4295.293,0.924,1,-460.0,-1624.0,140.0,OnTrack,1195,0.01195,64.145573
3,2025,Y2025R02S4,81,19,2025-03-22 07:58:41.020000+00:00,4295.513,1.144,1,-628.0,-1665.0,140.0,OnTrack,1517,0.01517,81.420107
4,2025,Y2025R02S4,81,19,2025-03-22 07:58:41.180000+00:00,4295.673,1.304,1,-751.0,-1695.0,140.0,OnTrack,1753,0.01753,94.080355
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
347,2025,Y2025R02S4,81,19,2025-03-22 08:00:09.361000+00:00,4383.854,89.485,1,1167.0,-1246.0,145.0,OnTrack,98082,0.98082,5264.085057
348,2025,Y2025R02S4,81,19,2025-03-22 08:00:09.681000+00:00,4384.174,89.805,1,945.0,-1293.0,141.0,OnTrack,98505,0.98505,5286.779169
349,2025,Y2025R02S4,81,19,2025-03-22 08:00:09.901000+00:00,4384.394,90.025,1,581.0,-1372.0,140.0,OnTrack,99199,0.99199,5324.040450
350,2025,Y2025R02S4,81,19,2025-03-22 08:00:10.320000+00:00,4384.813,90.444,1,369.0,-1420.0,141.0,OnTrack,99604,0.99604,5345.760655


In [19]:
_start_2026, _end_2026 = 3120, 4755
df1 = data_telemetry_pos_lap2026[
    (data_telemetry_pos_lap2026['distance_m'] > _start_2026 - 50)
    & (data_telemetry_pos_lap2026['distance_m'] < _end_2026 + 50)
]
df2 = data_telemetry_car_lap2026[
    (data_telemetry_car_lap2026['timing_from_lap'] > df1['timing_from_lap'].min())
    & (data_telemetry_car_lap2026['timing_from_lap'] < df1['timing_from_lap'].max())
]
df2['distance_m'] = np.interp(
    df2['timing_from_lap'],
    data_telemetry_pos_lap2026['timing_from_lap'],
    data_telemetry_pos_lap2026['distance_m'],
)

/var/folders/n9/qtf08bkx00z3xzx24pfblmhc0000gp/T/ipykernel_73695/3427446116.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['distance_m'] = np.interp(


In [20]:
_start_2025, _end_2025 = 3060, 4705
df3 = data_telemetry_pos_lap2025[
    (data_telemetry_pos_lap2025['distance_m'] > _start_2025 - 50)
    & (data_telemetry_pos_lap2025['distance_m'] < _end_2025 + 50)
]
df4 = data_telemetry_car_lap2025[
    (data_telemetry_car_lap2025['timing_from_lap'] > df3['timing_from_lap'].min())
    & (data_telemetry_car_lap2025['timing_from_lap'] < df3['timing_from_lap'].max())
]
df4['distance_m'] = np.interp(
    df4['timing_from_lap'],
    data_telemetry_pos_lap2025['timing_from_lap'],
    data_telemetry_pos_lap2025['distance_m'],
)

/var/folders/n9/qtf08bkx00z3xzx24pfblmhc0000gp/T/ipykernel_73695/3883582825.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df4['distance_m'] = np.interp(


In [21]:
plot_df = df2[(df2['distance_m']>_start_2026) & (df2['distance_m']<_end_2026)]
plot_df['distance_plt'] = plot_df['distance_m'] - _end_2026
plot_data = [
    go.Scatter(
        x=plot_df['distance_plt'].values,
        y=plot_df['speed'].values,
    )
]
plot_layout = dict(
    title="Telemetry data on the",
)
fig = go.Figure(data=plot_data, layout=plot_layout)
fig.show()

/var/folders/n9/qtf08bkx00z3xzx24pfblmhc0000gp/T/ipykernel_73695/4061306457.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  plot_df['distance_plt'] = plot_df['distance_m'] - _end_2026


In [22]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=["Speed", "Throttle"],
    # title="Telemetry data on the Shanghai straight",
)
plot_df = df2[(df2['distance_m']>_start_2026) & (df2['distance_m']<_end_2026 + 100)]
plot_df['distance_plt'] = plot_df['distance_m'] - _end_2026
fig.add_trace(
    go.Scatter(
        x=plot_df['distance_plt'].values,
        y=plot_df['speed'].values,
        name='Speed - 2026',
        line=dict(color='blue'),
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(
        x=plot_df['distance_plt'].values,
        y=plot_df['throttle'].values,
        name='Throttle - 2026',
        line=dict(color='blue'),
    ),
    row=2, col=1,
)
plot_df = df4[(df4['distance_m']>_start_2025) & (df4['distance_m']<_end_2025 + 100)]
plot_df['distance_plt'] = plot_df['distance_m'] - _end_2025
fig.add_trace(
    go.Scatter(
        x=plot_df['distance_plt'].values,
        y=plot_df['speed'].values,
        name='Speed - 2025',
        line=dict(color='green'),
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(
        x=plot_df['distance_plt'].values,
        y=plot_df['throttle'].values,
        name='Throttle - 2025',
        line=dict(color='green'),
    ),
    row=2, col=1,
)
fig.update_layout(
    title="Telemetry data on the Shanghai straight (T11 to T14)",
    height=600,
    width=1000,
    hovermode="x unified",
    showlegend=False,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.update_xaxes(
    showspikes=True,
    spikemode="across",
    spikedash="solid",
    spikecolor="rgba(128, 128, 128, 0.5)",
    spikethickness=1,
)



/var/folders/n9/qtf08bkx00z3xzx24pfblmhc0000gp/T/ipykernel_73695/3081301320.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  plot_df['distance_plt'] = plot_df['distance_m'] - _end_2026
/var/folders/n9/qtf08bkx00z3xzx24pfblmhc0000gp/T/ipykernel_73695/3081301320.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  plot_df['distance_plt'] = plot_df['distance_m'] - _end_2025
